In [ ]:
import os
import sys
import numpy as np
import pandas as pd

sys.path.append(os.path.abspath("../../")) ; from EPF import variables
sys.path.insert(0, "../helpers/"); from run_parrellel import run_parallel

Evaluate the model

In [ ]:
def evaluate(model_dict, features_optimal_amount):
    feature_data = pd.read_parquet(variables.FEATURES_DATASET_PATH)
    target_data  = pd.read_parquet(variables.AGG_TARGET_DATASET_PATH)
    _, _, test_feat = _split(feature_data)
    _, _, test_tgt = _split(target_data)

    horizon_list = model_dict["horizon_list"]
    n_h = len(horizon_list)
    base_models = model_dict.get("base_models", [None]*n_h)
    base_l2_models = model_dict.get("base_l2_models", [None]*n_h)
    blend_l2_alphas = model_dict.get("blend_l2_alphas", np.zeros(n_h, dtype=np.float32))
    spike_clfs = model_dict.get("spike_clfs", [None]*n_h)
    spike_regs = model_dict.get("spike_regs", [None]*n_h)
    spike_qregs = model_dict.get("spike_qregs", [None]*n_h)
    dip_clfs = model_dict.get("dip_clfs", [None]*n_h)
    dip_regs = model_dict.get("dip_regs", [None]*n_h)
    blend_alphas = model_dict.get("blend_alphas", np.ones(n_h,  dtype=np.float32))
    spike_kind = model_dict.get("spike_kind", np.zeros(n_h, dtype=np.int8))
    spike_src = model_dict.get("spike_src", np.zeros(n_h, dtype=np.int8))
    spike_p1 = model_dict.get("spike_p1", np.full(n_h, 0.20, dtype=np.float32))
    spike_p2 = model_dict.get("spike_p2", np.full(n_h, 2.0,  dtype=np.float32))
    spike_p3 = model_dict.get("spike_p3", np.full(n_h, 0.75, dtype=np.float32))
    dip_kind = model_dict.get("dip_kind", np.zeros(n_h, dtype=np.int8))
    dip_p1 = model_dict.get("dip_p1", np.full(n_h, 0.15, dtype=np.float32))
    dip_p2 = model_dict.get("dip_p2", np.full(n_h, 1.0,  dtype=np.float32))
    dip_p3 = model_dict.get("dip_p3", np.full(n_h, 0.60, dtype=np.float32))
    calibrators = model_dict.get("calibrators", [None]*n_h)

    naive_col = f"{variables.SELECTED_TARGET_COLUMN_NAME}_lag_336"
    n_rows = len(test_tgt)
    preds_m = np.full((n_rows, n_h), np.nan, dtype=np.float32)
    trues_m = np.full((n_rows, n_h), np.nan, dtype=np.float32)

    for i, h in enumerate(horizon_list):
        feat_cols = [c for c in features_optimal_amount.loc[features_optimal_amount[f"h{h}"] == True, "feature"].tolist() if c in test_feat.columns]
        X_te = test_feat.reindex(test_tgt.index)[feat_cols].values.astype(np.float32)

        y_base = _from_asinh(base_models[i].predict(X_te)).astype(np.float32)
        if base_l2_models[i] is not None:
            _la = float(blend_l2_alphas[i])
            y_base = ((1.0 - _la) * y_base + _la * _from_asinh(base_l2_models[i].predict(X_te))).astype(np.float32)

        y_model = y_base
        if spike_clfs[i] is not None:
            sp = spike_clfs[i].predict_proba(X_te)[:, 1].astype(np.float32)
            sr = _from_asinh(spike_regs[i].predict(X_te)).astype(np.float32) if spike_regs[i] else y_base
            sq = _from_asinh(spike_qregs[i].predict(X_te)).astype(np.float32) if spike_qregs[i] else sr
            y_model = _apply_spike_policy(y_base, sp, sr, sq, int(spike_kind[i]), float(spike_p1[i]), float(spike_p2[i]), float(spike_p3[i]), int(spike_src[i]))

        if dip_clfs[i] is not None and dip_regs[i] is not None:
            dp = dip_clfs[i].predict_proba(X_te)[:, 1].astype(np.float32)
            dr = _from_asinh(dip_regs[i].predict(X_te)).astype(np.float32)
            y_model = _apply_dip_policy(y_model, dp, dr, int(dip_kind[i]), float(dip_p1[i]), float(dip_p2[i]), float(dip_p3[i]))

        a = float(blend_alphas[i])
        if naive_col in test_feat.columns and a < 1.0:
            naive_h = test_feat.reindex(test_tgt.index)[naive_col].values.astype(np.float32)
            y_model = a * y_model + (1.0 - a) * naive_h

        if calibrators[i] is not None:
            y_model = calibrators[i].predict(y_model.astype(np.float64)).astype(np.float32)

        preds_m[:, i] = y_model
        trues_m[:, i] = test_tgt[f"target_h{h}"].values.astype(np.float32)

    valid_rows = np.all(np.isfinite(trues_m), axis=1)
    preds_m = preds_m[valid_rows]; trues_m = trues_m[valid_rows]
    test_idx = test_tgt.index[valid_rows]

    def _rmse(yt, yp): return float(np.sqrt(mean_squared_error(yt, yp)))
    def _wmape(yt, yp): return float(np.sum(np.abs(yt - yp)) / (np.sum(np.abs(yt)) + 1e-8) * 100)
    def _mbe(yt, yp): return float(np.mean(yp - yt))
    def _build_metrics(yt, yp):
        return {"mae": float(mean_absolute_error(yt, yp)), "rmse": _rmse(yt, yp), "r2": float(r2_score(yt, yp)), "mbe": _mbe(yt, yp), "wmape": _wmape(yt, yp)}

    yt_flat = trues_m.ravel(); yp_flat = preds_m.ravel()
    model_metrics = _build_metrics(yt_flat, yp_flat)
    spike_mask = yt_flat > variables.SPIKE_THRESHOLD
    if spike_mask.sum() > 0:
        model_metrics["spike_mae"]    = float(mean_absolute_error(yt_flat[spike_mask],  yp_flat[spike_mask]))
        model_metrics["nonspike_mae"] = float(mean_absolute_error(yt_flat[~spike_mask], yp_flat[~spike_mask]))
        model_metrics["spike_pct"]    = float(spike_mask.mean() * 100)
    dip_mask = yt_flat < variables.DIP_THRESHOLD
    if dip_mask.sum() > 0:
        model_metrics["dip_mae"] = float(mean_absolute_error(yt_flat[dip_mask], yp_flat[dip_mask]))
        model_metrics["dip_pct"] = float(dip_mask.mean() * 100)

    if naive_col in test_feat.columns:
        naive_vals = test_feat.reindex(test_tgt.index)[naive_col].values[valid_rows]
        naive_metrics = _build_metrics(trues_m.ravel(), np.repeat(naive_vals[:, None], n_h, axis=1).ravel())
    else:
        naive_metrics = model_metrics

    step_records = [{"step": i+1, "h": h,
                     "lead_h": round(h * variables.HORIZON_GRANULARITY_IN_MINUTES / 60, 1),
                     "mae":  round(float(mean_absolute_error(trues_m[:, i], preds_m[:, i])), 2),
                     "rmse": round(_rmse(trues_m[:, i], preds_m[:, i]), 2),
                     "r2":   round(float(r2_score(trues_m[:, i], preds_m[:, i])), 4),
                     "mbe":  round(_mbe(trues_m[:, i], preds_m[:, i]), 2)} for i, h in enumerate(horizon_list)]

    results_df = pd.DataFrame(index=test_idx)
    for i, h in enumerate(horizon_list):
        results_df[f"actual_h{h}"]    = trues_m[:, i]
        results_df[f"predicted_h{h}"] = preds_m[:, i]

    return {"model": model_metrics, "naive": naive_metrics, "results_df": results_df, "steps_df": pd.DataFrame(step_records)}

In [ ]:

eval_output = evaluate(model_dict, features_optimal_amount)
print_metrics(eval_output)